In [6]:
import finnhub
import os
import time
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

# Date range — aligns with EPS surprises window
FROM_DATE = "2023-01-01"
TO_DATE = "2025-12-31"

data = finnhub_client.stock_lobbying(symbol="AAPL", _from=FROM_DATE, to=TO_DATE)

In [7]:
# Top level keys
print("=== TOP LEVEL KEYS ===")
for key in data.keys():
    print(f"  {key}")

# How many lobbying activities returned
print(f"\nLobbying activities returned: {len(data['data'])}")

=== TOP LEVEL KEYS ===
  data
  symbol

Lobbying activities returned: 102


In [8]:
# Look at first activity in full
print("=== FIRST ACTIVITY ===")
for k, v in data["data"][0].items():
    print(f"  {k}: {v}")

=== FIRST ACTIVITY ===
  symbol: AAPL
  name: APPLE, INC.
  description: Technology company
  country: US
  year: 2024
  period: Q4
  documentUrl: https://lda.senate.gov/filings/public/filing/2c6c1a3d-546a-4e2e-b7cc-53f4eb432c97/print/
  income: 40000
  expenses: None
  postedName: 
  date: 
  clientId: 63242
  registrantId: 401109033
  senateId: 401109033-63242
  houseRegistrantId: 


In [9]:
# Understand the range of expenses and dates
print("=== EXPENSES ACROSS ALL ACTIVITIES ===")
for activity in data["data"]:
    print(f"  {activity['date']} | {activity['period']} | expenses: {activity['expenses']} | income: {activity['income']}")

=== EXPENSES ACROSS ALL ACTIVITIES ===
   | Q4 | expenses: None | income: 40000
   | Q1 | expenses: None | income: 60000
   | Q2 | expenses: None | income: 60000
   | Q3 | expenses: None | income: 60000
   | Q3 | expenses: None | income: 50000
   | Q3 | expenses: None | income: 20000
   | Q4 | expenses: 2870000 | income: None
   | Q1 | expenses: 2660000 | income: None
   | Q2 | expenses: 2410000 | income: None
   | Q3 | expenses: 1840000 | income: None
   | Q4 | expenses: 2950000 | income: None
   | Q1 | expenses: 2130000 | income: None
   | Q2 | expenses: 2130000 | income: None
   | Q3 | expenses: 1900000 | income: None
   | Q4 | expenses: 1660000 | income: None
   | Q1 | expenses: 2450000 | income: None
   | Q2 | expenses: 2290000 | income: None
   | Q3 | expenses: 2530000 | income: None
   | Q4 | expenses: None | income: 90000
   | Q1 | expenses: None | income: 90000
   | Q2 | expenses: None | income: 90000
   | Q3 | expenses: None | income: 90000
   | Q4 | expenses: None | income: 

### ─────────────────────────────────────────────
### SECTION 2 — PRODUCTION RUN (ALL 60 COMPANIES)
### ─────────────────────────────────────────────

In [10]:
import finnhub
import os
import time
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

FROM_DATE = "2023-01-01"
TO_DATE = "2025-12-31"

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

# Duplicate guard
duplicates = [t for t in tickers if tickers.count(t) > 1]
assert not duplicates, f"Duplicate tickers found: {duplicates}"

results = {}
issues = []

for i, ticker in enumerate(tickers):
    try:
        data = finnhub_client.stock_lobbying(symbol=ticker, _from=FROM_DATE, to=TO_DATE)
        results[ticker] = data

        # Check for empty response
        if not data or not data.get("data"):
            issues.append((ticker, "no lobbying activities found in date range"))
            continue

        # Check for None fields in each activity
        for activity in data["data"]:
            none_fields = [k for k, v in activity.items() if v is None or v == ""]
            if none_fields:
                issues.append((ticker, f"missing fields in activity {activity.get('date')}: {none_fields}"))
                break  # flag once per company, not per activity

    except Exception as e:
        issues.append((ticker, f"API error: {str(e)}"))
        results[ticker] = {}

    if i < len(tickers) - 1:
        time.sleep(2) # increased from 1 to 2 for this endpoint

# Summary
successful = [t for t, r in results.items() if r and r.get("data")]
no_activity = [t for t, r in results.items() if r is not None and not r.get("data")]

print(f"✅ Successfully pulled: {len(results)} / {len(tickers)} companies")
print(f"📋 Companies with lobbying activity: {len(successful)}")
print(f"⭕ Companies with no lobbying activity: {len(no_activity)}")
print(f"   {no_activity}")
print(f"⚠️  Issues found: {len(issues)}")
for ticker, issue in issues:
    print(f"   {ticker}: {issue}")

✅ Successfully pulled: 60 / 60 companies
📋 Companies with lobbying activity: 46
⭕ Companies with no lobbying activity: 14
   ['BAESY', 'HEI', 'DRS', 'DCO', 'BP', 'SLB', 'EOG', 'FANG', 'CTRA', 'AR', 'CHRD', 'MTDR', 'IBM', 'DDOG']
⚠️  Issues found: 60
   LMT: missing fields in activity : ['expenses', 'postedName', 'date']
   RTX: missing fields in activity : ['expenses', 'postedName', 'date']
   NOC: missing fields in activity : ['expenses', 'postedName', 'date']
   GD: missing fields in activity : ['expenses', 'postedName', 'date']
   BA: missing fields in activity : ['expenses', 'postedName', 'date']
   LHX: missing fields in activity : ['expenses', 'postedName', 'date']
   LDOS: missing fields in activity : ['expenses', 'postedName', 'date']
   HII: missing fields in activity : ['expenses', 'postedName', 'date']
   BAESY: no lobbying activities found in date range
   SAIC: missing fields in activity : ['expenses', 'postedName', 'date']
   TXT: missing fields in activity : ['expenses',